6BUIS017W CRM & CM With Business Intelligence – Individual
Coursework-CRM Case Study

In [33]:
import pandas as pd
import yfinance as yf
from datetime import datetime
import requests
import io

*Y finance was chosen as I believe its best because it provides free, API-key-free access to global market data directly as pandas DataFrames, making my Python analysis fast*

Survivorship bias note: This analysis uses the current (2026) S&P 500 constituent list applied retroactively to 2021–2025 price history, meaning companies that were removed from the index (via delisting, acquisition, or bankruptcy) during the study period are excluded; this biases the sample toward firms that survived and likely performed relatively well, so results should not be read as representative of the full historical index population.

In [34]:
retrieval_date = datetime.now().strftime("%Y-%m-%d")

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36"
}

url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
response = requests.get(url, headers=headers)
response.raise_for_status()

tables = pd.read_html(io.StringIO(response.text))
sp500 = tables[0]
tickers = sp500['Symbol'].str.replace('.', '-', regex=False).tolist()

print(f"Retrieved {len(tickers)} tickers on {retrieval_date}")
sp500.to_csv("sp500_constituents_snapshot.csv", index=False)

Retrieved 503 tickers on 2026-11-23


The Wikipedia constituent table returned 503 tickers rather than exactly 500, because a small number of S&P 500 companies (e.g. Alphabet, Fox Corporation) have dual share classes listed as separate symbols (such as GOOGL/GOOG), which count as distinct tickers despite representing a single index constituent.

In [35]:
# -------------------------------
# Step 2: Download daily price/volume data per ticker (with retry)
# -------------------------------
data = {}
failed = []

for t in tickers:
    success = False
    for attempt in range(3):
        try:
            df = yf.download(t, start="2021-01-01", end="2025-01-01", auto_adjust=False, progress=False)
            if not df.empty:
                data[t] = df
                success = True
                break
            else:
                time.sleep(1)
        except Exception:
            time.sleep(1)
    if not success:
        failed.append(t)
    time.sleep(0.1)

print(f"Downloaded {len(data)} tickers, {len(failed)} failed")
print(f"Failed tickers: {failed}")

# -------------------------------
# Step 3: Download benchmark (^GSPC)
# -------------------------------
benchmark = yf.download("^GSPC", start="2021-01-01", end="2025-01-01", auto_adjust=False, progress=False)

# -------------------------------
# Step 4: Cache everything immediately (fallback if Colab disconnects)
# -------------------------------

import pickle

benchmark = yf.download("^GSPC", start="2021-01-01", end="2025-01-01", auto_adjust=False, progress=False)

with open("price_data_cache.pkl", "wb") as f:
    pickle.dump({
        "data": data,
        "benchmark": benchmark,
        "failed": failed,
        "retrieval_date": retrieval_date
    }, f)

print("Cached to price_data_cache.pkl")
with open("price_data_cache.pkl", "wb") as f:
    pickle.dump({
        "data": data,
        "benchmark": benchmark,
        "failed": failed,
        "retrieval_date": retrieval_date
    }, f)

print("Cached to price_data_cache.pkl")

ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['FDXF']: YFPricesMissingError('possibly delisted; no price data found  (1d 2021-01-01 -> 2025-01-01) (Yahoo error = "Data doesn\'t exist for startDate = 1609477200, endDate = 1735707600")')
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['FDXF']: YFPricesMissingError('possibly delisted; no price data found  (1d 2021-01-01 -> 2025-01-01) (Yahoo error = "Data doesn\'t exist for startDate = 1609477200, endDate = 1735707600")')
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['FDXF']: YFPricesMissingError('possibly delisted; no price data found  (1d 2021-01-01 -> 2025-01-01) (Yahoo error = "Data doesn\'t exist for startDate = 1609477200, endDate = 1735707600")')
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['HONA']: YFPricesMissingError('possibly delisted; no price data found  (1d 2021-01-01 -> 2025-01-01) (Yahoo error = "Data doesn\'t exist for startDate = 1609477200, endDate = 1735707600")')
ERROR:yfinance:
1 Failed download:
E

Downloaded 498 tickers, 5 failed
Failed tickers: ['FDXF', 'HONA', 'Q', 'SNDK', 'VYLR']
Cached to price_data_cache.pkl
Cached to price_data_cache.pkl


498 tickers of the 503 retrieved had no retrievable price history for the full 2021–2025 window, most likely due to recent index additions, spin-offs, or ticker symbol changes rather than true delisting — these were excluded from the dataset and logged for transparency in Section 1.2.

In [36]:
## 1.2

In [37]:
for t in data:
    if isinstance(data[t].columns, pd.MultiIndex):
        data[t].columns = data[t].columns.get_level_values(0)

exclusion_log = []

def validate_ticker(t, df, min_days=900):
    issues = []
    if df.index.duplicated().any():
        issues.append("duplicate dates")
    if df[['Open','High','Low','Close','Adj Close']].isna().any().any():
        issues.append("missing values")
    if (df['Close'] <= 0).any() or (df['Adj Close'] <= 0).any():
        issues.append("nonpositive price")
    if (df['Volume'] < 0).any():
        issues.append("negative volume")
    if len(df) < min_days:
        issues.append(f"short history ({len(df)} days)")
    return issues

valid_data = {}
for t, df in data.items():
    issues = validate_ticker(t, df)
    if issues:
        exclusion_log.append({"ticker": t, "issues": "; ".join(issues)})
    else:
        valid_data[t] = df

for t in failed:
    exclusion_log.append({"ticker": t, "issues": "download failed — no data in period (recent IPO/ticker change/delisted)"})

exclusion_df = pd.DataFrame(exclusion_log)
exclusion_df.to_csv("exclusion_log.csv", index=False)

print(f"Started with: {len(data)} downloaded tickers")
print(f"Valid tickers after validation: {len(valid_data)}")
print(f"Total excluded (failed + validation issues): {len(exclusion_log)}")
exclusion_df

Started with: 498 downloaded tickers
Valid tickers after validation: 491
Total excluded (failed + validation issues): 12


,ticker,issues
0,CEG,short history (742 days)
1,GEHC,short history (513 days)
2,GEV,short history (193 days)
3,RDDT,short history (197 days)
4,HOOD,short history (862 days)
5,SOLV,short history (194 days)
6,VLTO,short history (313 days)
7,FDXF,download failed — no data in period (recent IP...
8,HONA,download failed — no data in period (recent IP...
9,Q,download failed — no data in period (recent IP...


Seven tickers (CEG, GEHC, GEV, RDDT, HOOD, SOLV, VLTO) were excluded due to insufficient price history, as each represents a company that IPO'd or was spun off from its parent company after the study period began (2021–2024), rather than a data quality defect. A minimum of 900 trading days (~3.5 years) was required for inclusion to ensure sufficient observations for stable weekly beta estimation.

In [38]:
## 1.3

In [39]:
weekly_prices = {}
for t, df in valid_data.items():
    w = df['Adj Close'].resample('W-FRI').last()
    weekly_prices[t] = w

price_matrix = pd.DataFrame(weekly_prices)

# Check for partial weeks at the edges before trimming
print("First week label:", price_matrix.index[0])
print("Last week label:", price_matrix.index[-1])

First week label: 2021-01-08 00:00:00
Last week label: 2025-01-03 00:00:00


In [40]:
return_matrix = price_matrix.pct_change()
return_matrix = return_matrix.iloc[1:-1]

mean_weekly_return = return_matrix.mean()
print(f"Weekly return matrix shape: {return_matrix.shape}")
mean_weekly_return.head()

Weekly return matrix shape: (207, 491)


,0
MMM,0.001142
AOS,0.001991
ABT,0.000971
ABBV,0.003749
ACN,0.002376


In [41]:
if isinstance(benchmark.columns, pd.MultiIndex):
    benchmark.columns = benchmark.columns.get_level_values(0)

benchmark_weekly_price = benchmark['Adj Close'].resample('W-FRI').last()
benchmark_weekly_return = benchmark_weekly_price.pct_change().iloc[1:-1]

print(f"Benchmark weekly return observations: {len(benchmark_weekly_return)}")

Benchmark weekly return observations: 207


In [42]:
# 1.3.2

In [43]:
betas = {}
obs_counts = {}

for t in return_matrix.columns:
    aligned = pd.concat([return_matrix[t], benchmark_weekly_return], axis=1).dropna()
    aligned.columns = ['share', 'bench']

    if len(aligned) < 2:
        continue

    cov = aligned['share'].cov(aligned['bench'])
    var = aligned['bench'].var()
    betas[t] = cov / var
    obs_counts[t] = len(aligned)

beta_series = pd.Series(betas)
obs_count_series = pd.Series(obs_counts)

print(f"Betas computed for {len(beta_series)} tickers")
print(f"Observation counts — min: {obs_count_series.min()}, max: {obs_count_series.max()}, mode: {obs_count_series.mode()[0]}")
beta_series.describe()

Betas computed for 491 tickers
Observation counts — min: 193, max: 207, mode: 207


,0
count,491.000000
mean,0.980713
std,0.420748
min,0.055453
25%,0.668916
50%,0.953177
75%,1.180578
max,3.778560


In [44]:
example_ticker = "AAPL"

print(f"Mean weekly return for {example_ticker}: {mean_weekly_return[example_ticker]:.5f}")
print(f"Weekly beta for {example_ticker}: {beta_series[example_ticker]:.4f}")
print(f"Observations used: {obs_counts[example_ticker]}")

Mean weekly return for AAPL: 0.00393
Weekly beta for AAPL: 1.1748
Observations used: 207


In [45]:
## 1.4

In [46]:
# Flatten MultiIndex columns for valid_data too, if not already done
for t in valid_data:
    if isinstance(valid_data[t].columns, pd.MultiIndex):
        valid_data[t].columns = valid_data[t].columns.get_level_values(0)

# Observation date = latest date common to ALL valid tickers
observation_date = min(df.index.max() for df in valid_data.values())
print(f"Observation date: {observation_date}")

Observation date: 2024-12-31 00:00:00


In [47]:
def compute_rfm(df, obs_date, window=60, baseline=20):
    df = df.sort_index()
    df = df[df.index <= obs_date]

    if len(df) < window + baseline:
        return None, None, None  # not enough history to compute reliably

    last_60 = df.iloc[-window:]

    qualifying_days = []
    for i in range(len(last_60)):
        current_date = last_60.index[i]
        hist = df[df.index < current_date].iloc[-baseline:]
        if len(hist) < baseline:
            continue
        threshold = 2 * hist['Volume'].mean()
        if last_60['Volume'].iloc[i] > threshold:
            qualifying_days.append(current_date)

    # Recency: calendar days since latest qualifying day
    if qualifying_days:
        recency = (obs_date - max(qualifying_days)).days
    else:
        recency = None

    # Frequency: count of qualifying days
    frequency = len(qualifying_days)

    # Monetary: sum of Close * Volume over the 60 sessions
    monetary = (last_60['Close'] * last_60['Volume']).sum()

    return recency, frequency, monetary

In [48]:
rfm_rows = []
for t, df in valid_data.items():
    r, f, m = compute_rfm(df, observation_date)
    rfm_rows.append({"ticker": t, "R": r, "F": f, "M": m})

rfm_df = pd.DataFrame(rfm_rows).set_index("ticker")

print(f"Total tickers: {len(rfm_df)}")
print(f"Tickers with no qualifying high-volume day (R = missing): {rfm_df['R'].isna().sum()}")
rfm_df.describe()

Total tickers: 491
Tickers with no qualifying high-volume day (R = missing): 10


,R,F,M
count,481.000000,491.000000,4.910000e+02
mean,18.430353,2.993890,3.973205e+10
std,15.302091,1.328634,1.287707e+11
min,4.000000,0.000000,1.297483e+09
25%,11.000000,2.000000,1.025172e+10
50%,11.000000,3.000000,1.670905e+10
75%,13.000000,4.000000,3.242874e+10
max,85.000000,8.000000,1.847697e+12


In [49]:
rfm_df.sort_values("M", ascending=False).head(5)

,R,F,M
ticker,,,
TSLA,50.0,5,1.847697e+12
NVDA,NaN,0,1.825337e+12
AAPL,11.0,2,6.233034e+11
MSFT,11.0,2,5.348982e+11
AMZN,11.0,5,4.681770e+11


In [50]:
print(f"Recency: {rfm_df.loc['AAPL', 'R']}")
print(f"Frequency: {rfm_df.loc['AAPL', 'F']}")
print(f"Monetary: ${rfm_df.loc['AAPL', 'M']:,.0f}")

Recency: 11.0
Frequency: 2
Monetary: $623,303,377,707


In [51]:
rfm_df.to_csv("rfm_features.csv")
mean_weekly_return.to_csv("mean_weekly_return.csv")
beta_series.to_csv("weekly_beta.csv")
exclusion_df.to_csv("exclusion_log.csv", index=False)

In [52]:
print("hi")



hi
